# Etapa 6 — Avaliação dos Modelos

**Objetivo:** Calcular métricas finais, analisar resíduos do melhor modelo e gerar todos os gráficos comparativos.

**Métricas:** MAE, RMSE, sMAPE (Makridakis, 1993)

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

plt.rcParams.update({'figure.dpi': 150, 'font.size': 10,
                     'axes.titlesize': 11, 'axes.labelsize': 10})

PROCESSED_DIR = '../data/processed/'
FIGURES_DIR   = '../outputs/figures/'
TABLES_DIR    = '../outputs/tables/'

pred  = pd.read_csv(PROCESSED_DIR + 'predicoes_teste.csv', parse_dates=['date'])
daily = pd.read_csv(PROCESSED_DIR + 'beverages_daily.csv', parse_dates=['date'])
imp   = pd.read_csv(PROCESSED_DIR + 'xgb_feature_importance.csv')
comp  = pd.read_csv(PROCESSED_DIR + 'prophet_components.csv', parse_dates=['ds'])

print(f'Previsões: {pred.shape}  |  Período: {pred.date.min().date()} a {pred.date.max().date()}')

Previsões: (90, 7)  |  Período: 2017-05-18 a 2017-08-15


## 6.1 Métricas finais

In [2]:
def mae(y, yhat):   return np.mean(np.abs(y - yhat))
def rmse(y, yhat):  return np.sqrt(np.mean((y - yhat) ** 2))
def smape(y, yhat): return np.mean(np.abs(y - yhat) / ((np.abs(y) + np.abs(yhat)) / 2)) * 100

y = pred['y_real'].values
modelos = ['naive', 'sazonal_naive', 'sarima', 'prophet', 'xgboost']
labels  = ['Naive', 'Sazonal Naive', 'SARIMA (1,1,1)(1,1,1,7)', 'Prophet', 'XGBoost']

rows = []
for col, label in zip(modelos, labels):
    yhat = pred[col].values
    rows.append({'Modelo': label, 'MAE': mae(y,yhat), 'RMSE': rmse(y,yhat), 'sMAPE (%)': smape(y,yhat)})

metrics_df = pd.DataFrame(rows).sort_values('MAE').reset_index(drop=True)
print(metrics_df.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

metrics_df.to_csv(TABLES_DIR + 'metricas_comparativas.csv', index=False)
print('\nSalvo: metricas_comparativas.csv')

                 Modelo      MAE     RMSE  sMAPE (%)
                XGBoost 12944.28 18505.50       6.40
                Prophet 18483.69 24768.05       9.39
          Sazonal Naive 21801.26 28284.83      10.80
SARIMA (1,1,1)(1,1,1,7) 35336.31 43418.59      20.76
                  Naive 36748.08 53082.55      18.58

Salvo: metricas_comparativas.csv


## 6.2 Real vs. Previsto — separados por modelo (5 gráficos)

In [3]:
cores = {'naive': '#e74c3c', 'sazonal_naive': '#e67e22',
         'sarima': '#9b59b6', 'prophet': '#2980b9', 'xgboost': '#27ae60'}

for col, label in zip(modelos, labels):
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(pred['date'], pred['y_real'], color='#2c3e50', linewidth=1.5,
            label='Real', zorder=3)
    ax.plot(pred['date'], pred[col], color=cores[col], linewidth=1.4,
            linestyle='--', label=f'Previsto ({label})', zorder=2)
    ax.set_title(f'{label} — Real vs. Previsto (hold-out 90 dias)')
    ax.set_xlabel('Data')
    ax.set_ylabel('Vendas (unidades)')
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    plt.xticks(rotation=45)
    plt.tight_layout()
    fname = f'09_{col}_real_vs_previsto.png'
    plt.savefig(FIGURES_DIR + fname, bbox_inches='tight')
    plt.close()
    print(f'Salvo: {fname}')

Salvo: 09_naive_real_vs_previsto.png
Salvo: 09_sazonal_naive_real_vs_previsto.png


Salvo: 09_sarima_real_vs_previsto.png
Salvo: 09_prophet_real_vs_previsto.png


Salvo: 09_xgboost_real_vs_previsto.png


## 6.3 Overlay — todos os modelos sobrepostos

In [4]:
estilos = ['-', '--', '-.', ':', '--']

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(pred['date'], pred['y_real'], color='#2c3e50', linewidth=2,
        label='Real', zorder=5)
for (col, label), estilo in zip(zip(modelos, labels), estilos):
    ax.plot(pred['date'], pred[col], color=cores[col], linewidth=1.2,
            linestyle=estilo, label=label, alpha=0.85)

ax.set_title('Comparação de Todos os Modelos — Hold-out 90 Dias (BEVERAGES)')
ax.set_xlabel('Data')
ax.set_ylabel('Vendas (unidades)')
ax.legend(loc='upper left', fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_DIR + '10_todos_modelos_sobrepostos.png', bbox_inches='tight')
plt.close()
print('Salvo: 10_todos_modelos_sobrepostos.png')

Salvo: 10_todos_modelos_sobrepostos.png


## 6.4 Resíduos do XGBoost (melhor modelo)

In [5]:
residuos = pred['y_real'] - pred['xgboost']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(pred['date'], residuos, color='steelblue', linewidth=0.9)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_title('Resíduos XGBoost ao Longo do Tempo')
axes[0].set_xlabel('Data')
axes[0].set_ylabel('Erro (real − previsto)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
axes[0].xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)

axes[1].hist(residuos, bins=25, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1)
axes[1].axvline(residuos.mean(), color='orange', linestyle='--', linewidth=1,
                label=f'Média: {residuos.mean():.0f}')
axes[1].set_title('Distribuição dos Resíduos XGBoost')
axes[1].set_xlabel('Erro')
axes[1].set_ylabel('Frequência')
axes[1].legend(fontsize=9)

stats.probplot(residuos, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot Resíduos XGBoost')

plt.tight_layout()
plt.savefig(FIGURES_DIR + '11_residuos_xgboost.png', bbox_inches='tight')
plt.close()
print('Salvo: 11_residuos_xgboost.png')

print(f'\nEstatísticas dos resíduos:')
print(f'  Média (viés):  {residuos.mean():.0f}')
print(f'  Desvio padrão: {residuos.std():.0f}')
print(f'  Mín / Máx:     {residuos.min():.0f} / {residuos.max():.0f}')

Salvo: 11_residuos_xgboost.png

Estatísticas dos resíduos:
  Média (viés):  4451
  Desvio padrão: 18063
  Mín / Máx:     -55646 / 67838


In [6]:
# ACF dos resíduos
fig, ax = plt.subplots(figsize=(10, 3))
plot_acf(residuos, lags=30, ax=ax, color='steelblue')
ax.set_title('ACF dos Resíduos XGBoost (lags 0–30)')
ax.set_xlabel('Lag')
ax.set_ylabel('Autocorrelação')
plt.tight_layout()
plt.savefig(FIGURES_DIR + '11b_residuos_acf.png', bbox_inches='tight')
plt.close()
print('Salvo: 11b_residuos_acf.png')

Salvo: 11b_residuos_acf.png


In [7]:
# Top 5 maiores erros absolutos
pred_anot = pred.copy()
pred_anot['erro_abs'] = np.abs(pred_anot['y_real'] - pred_anot['xgboost'])
pred_anot['erro_rel'] = (pred_anot['y_real'] - pred_anot['xgboost'])
top5_erros = pred_anot.nlargest(5, 'erro_abs')[['date','y_real','xgboost','erro_rel','erro_abs']]
top5_erros['dia_semana'] = pd.to_datetime(top5_erros['date']).dt.day_name()

print('Top 5 maiores erros absolutos do XGBoost:')
print(top5_erros.to_string(index=False, float_format=lambda x: f'{x:.0f}'))

Top 5 maiores erros absolutos do XGBoost:
      date  y_real  xgboost  erro_rel  erro_abs dia_semana
2017-06-11  311184   243346     67838     67838     Sunday
2017-08-13  202354   258000    -55646     55646     Sunday
2017-06-04  339352   284273     55079     55079     Sunday
2017-08-12  182318   232792    -50474     50474   Saturday
2017-05-21  280849   234681     46168     46168     Sunday


## 6.5 Feature Importance XGBoost (top 10)

In [8]:
top10 = imp.head(10).sort_values('importance')

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(top10['feature'], top10['importance'], color='steelblue', edgecolor='white')
for bar, val in zip(bars, top10['importance']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
ax.set_title('Feature Importance XGBoost — Top 10 (gain-based)')
ax.set_xlabel('Importância relativa')
ax.set_xlim(0, top10['importance'].max() * 1.18)
plt.tight_layout()
plt.savefig(FIGURES_DIR + '12_xgboost_feature_importance.png', bbox_inches='tight')
plt.close()
print('Salvo: 12_xgboost_feature_importance.png')

Salvo: 12_xgboost_feature_importance.png


## 6.6 Componentes do Prophet

In [9]:
weekly_col  = [c for c in comp.columns if c == 'weekly'][0] if 'weekly' in comp.columns else None
yearly_col  = [c for c in comp.columns if c == 'yearly'][0] if 'yearly' in comp.columns else None

print(f'Colunas disponíveis nos componentes Prophet: {list(comp.columns)[:15]}')
print(f'weekly: {weekly_col}  |  yearly: {yearly_col}')

Colunas disponíveis nos componentes Prophet: ['ds', 'trend', 'yhat_lower', 'yhat_upper', 'trend_lower', 'trend_upper', 'Batalla de Pichincha', 'Batalla de Pichincha_lower', 'Batalla de Pichincha_upper', 'Black Friday', 'Black Friday_lower', 'Black Friday_upper', 'Carnaval', 'Carnaval_lower', 'Carnaval_upper']
weekly: weekly  |  yearly: yearly


In [10]:
fig, axes = plt.subplots(4, 1, figsize=(14, 13))

# Tendência
axes[0].plot(comp['ds'], comp['trend'], color='steelblue', linewidth=1)
axes[0].set_title('Tendência')
axes[0].set_ylabel('Contribuição (unidades)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[0].xaxis.set_major_locator(mdates.YearLocator())

# Sazonalidade semanal — extrair 7 dias de referência para barra
if weekly_col:
    # Agrupar por dia da semana (0=Seg,...,6=Dom)
    comp_w = comp.copy()
    comp_w['dow'] = comp_w['ds'].dt.dayofweek
    weekly_avg = comp_w.groupby('dow')[weekly_col].mean()
    dow_labels = ['Seg','Ter','Qua','Qui','Sex','Sáb','Dom']
    axes[1].bar(range(7), weekly_avg.values, color='steelblue', edgecolor='white')
    axes[1].set_xticks(range(7))
    axes[1].set_xticklabels(dow_labels)
    axes[1].axhline(0, color='gray', linewidth=0.8)
    axes[1].set_title('Sazonalidade Semanal (média por dia da semana)')
    axes[1].set_ylabel('Contribuição (unidades)')

# Sazonalidade anual — usar 2016 como referência
if yearly_col:
    year_ref = comp[(comp['ds'].dt.year == 2016)].copy()
    axes[2].plot(year_ref['ds'], year_ref[yearly_col], color='steelblue', linewidth=1)
    axes[2].axhline(0, color='gray', linewidth=0.8)
    axes[2].set_title('Sazonalidade Anual (ano 2016 como referência)')
    axes[2].set_ylabel('Contribuição (unidades)')
    axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    axes[2].xaxis.set_major_locator(mdates.MonthLocator())

# Efeito de feriados — colunas que representam efeitos aditivos residuais
excluir = {'ds','trend','yhat','yhat_lower','yhat_upper','trend_lower','trend_upper',
           'additive_terms','additive_terms_lower','additive_terms_upper',
           'multiplicative_terms','multiplicative_terms_lower','multiplicative_terms_upper',
           'onpromotion','oil_price','extra_regressors_additive',
           'extra_regressors_additive_lower','extra_regressors_additive_upper',
           'weekly','yearly','weekly_lower','weekly_upper','yearly_lower','yearly_upper'}
holiday_cols = [c for c in comp.columns if c not in excluir
                and 'lower' not in c and 'upper' not in c]
if holiday_cols:
    hol_effect = comp[holiday_cols].sum(axis=1)
    nz = hol_effect != 0
    axes[3].bar(comp.loc[nz,'ds'], hol_effect[nz], color='salmon', edgecolor='white', width=1)
    axes[3].axhline(0, color='gray', linewidth=0.8)
    axes[3].set_title('Efeito de Feriados e Eventos (incluindo terremoto 04/2016)')
    axes[3].set_ylabel('Contribuição (unidades)')
    axes[3].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    axes[3].xaxis.set_major_locator(mdates.YearLocator())

plt.suptitle('Componentes do Prophet — BEVERAGES (2013–2017)', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR + '13_prophet_componentes.png', bbox_inches='tight')
plt.close()
print('Salvo: 13_prophet_componentes.png')

Salvo: 13_prophet_componentes.png


## 6.7 Inventário final

In [11]:
import os

figures = sorted(os.listdir(FIGURES_DIR))
tables  = sorted(os.listdir(TABLES_DIR))

print(f'outputs/figures/  ({len(figures)} arquivos):')
for f in figures:
    size_kb = os.path.getsize(FIGURES_DIR + f) / 1024
    print(f'  {f}  ({size_kb:.0f} KB)')

print(f'\noutputs/tables/  ({len(tables)} arquivos):')
for f in tables:
    print(f'  {f}')

print('\n=== MÉTRICAS FINAIS ===')
print(metrics_df.to_string(index=False))

print('\n=== TOP 5 ERROS ABSOLUTOS (XGBoost) ===')
print(top5_erros.to_string(index=False, float_format=lambda x: f'{x:.0f}'))

outputs/figures/  (18 arquivos):
  01_serie_diaria.png  (220 KB)
  02_serie_mensal.png  (96 KB)
  03_histograma_vendas.png  (77 KB)
  04_boxplot_sazonalidade.png  (93 KB)
  05_decomposicao_aditiva.png  (360 KB)
  06_decomposicao_multiplicativa.png  (374 KB)
  07_impacto_terremoto.png  (130 KB)
  08_decomposicao_anual.png  (242 KB)
  09_naive_real_vs_previsto.png  (123 KB)
  09_prophet_real_vs_previsto.png  (157 KB)
  09_sarima_real_vs_previsto.png  (165 KB)
  09_sazonal_naive_real_vs_previsto.png  (182 KB)
  09_xgboost_real_vs_previsto.png  (166 KB)
  10_todos_modelos_sobrepostos.png  (319 KB)
  11_residuos_xgboost.png  (125 KB)
  11b_residuos_acf.png  (31 KB)
  12_xgboost_feature_importance.png  (59 KB)
  13_prophet_componentes.png  (212 KB)

outputs/tables/  (1 arquivos):
  metricas_comparativas.csv

=== MÉTRICAS FINAIS ===
                 Modelo          MAE         RMSE  sMAPE (%)
                XGBoost 12944.279889 18505.504343   6.399030
                Prophet 18483.692424 247